# AIML-04 · Fine-tune YOLO on GPU with Weights & Biases (Colab / Kaggle)

CPU fine-tuning on a laptop is very slow, so do the real training here on a free GPU, then copy `best.pt` back to
`models/detection/` on your laptop.

**Before you start**
1. On your laptop: `python scripts/prepare_dataset.py ...` (see `docs/DATASETS.md`), then zip the folder `datasets/processed/<name>_yolo` and upload the zip to Google Drive (or Kaggle datasets).
2. Put your W&B API key in Colab *Secrets* (key icon on the left) with the name `WANDB_API_KEY`. **Never paste a key into a cell or commit it.**

> Status: *written but not executed by the author of the ZIP*. Your own run is the evidence for the fine-tuning claim.

In [ ]:
!nvidia-smi
!pip -q install ultralytics wandb

In [ ]:
import os
try:
    from google.colab import userdata, drive
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")   # stored in Colab Secrets, never in the notebook
    drive.mount("/content/drive")
except Exception as e:
    print("Not on Colab or secret missing:", e, "\n(on Kaggle use Add-ons -> Secrets and os.environ['WANDB_API_KEY'] = ...)")

In [ ]:
# EDIT THESE
DATASET_ZIP = "/content/drive/MyDrive/bdd100k_yolo.zip"   # the zip you uploaded
RUN_NAME    = "yolo11n_bdd100k_colab"
EPOCHS, IMGSZ, BATCH = 50, 640, 32

!mkdir -p /content/data && unzip -q -o "$DATASET_ZIP" -d /content/data
!ls /content/data

In [ ]:
import glob, yaml
cand = glob.glob("/content/data/**/data.yaml", recursive=True)
assert cand, "data.yaml not found in the zip"
data_yaml = cand[0]
cfg = yaml.safe_load(open(data_yaml)); cfg["path"] = os.path.dirname(data_yaml)   # fix the absolute Windows path inside data.yaml
yaml.safe_dump(cfg, open(data_yaml, "w"))
print(open(data_yaml).read())

In [ ]:
import wandb
from ultralytics import YOLO, settings
settings.update({"wandb": True})          # ultralytics' built-in W&B logging (metrics, curves, validation images)
model = YOLO("yolo11n.pt")
model.train(data=data_yaml, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0, project="aiml04", name=RUN_NAME, patience=15, plots=True)

In [ ]:
best = f"aiml04/{RUN_NAME}/weights/best.pt"
metrics = YOLO(best).val(data=data_yaml, imgsz=IMGSZ, device=0)
print("mAP50-95:", metrics.box.map, "mAP50:", metrics.box.map50, "P:", metrics.box.mp, "R:", metrics.box.mr)

In [ ]:
from google.colab import files
files.download(best)   # copy to models/detection/ on your laptop, then set detector.weights in configs/default.yaml

After downloading: `python scripts/evaluate_detector.py --data <data.yaml> --weights models/detection/best.pt --tag finetuned`
and `python scripts/export_onnx.py --weights models/detection/best.pt`. Only then can the report say the model was fine-tuned, and the numbers must be the ones you measured.